In [27]:
import pandas as pd

customers = pd.read_csv("customers_raw.csv")
applications = pd.read_csv("applications_raw.csv")
credit = pd.read_csv("credit_assessments_raw.csv")
loans = pd.read_csv("loans_raw.csv")
repayments = pd.read_csv("repayments_raw.csv")
collections = pd.read_csv("collections_raw.csv")
loan_status_history = pd.read_csv("loan_status_history_raw.csv")

In [28]:
# Remove duplicate records using primary keys

customers_clean = customers.drop_duplicates(subset="customer_id").copy()

applications_clean = applications.drop_duplicates(subset="application_id").copy()

repayments_clean = repayments.drop_duplicates(subset="repayment_id").copy()


# Check before and after row counts

print("Customers:", customers.shape, "→", customers_clean.shape)
print("Applications:", applications.shape, "→", applications_clean.shape)
print("Repayments:", repayments.shape, "→", repayments_clean.shape)

Customers: (50100, 9) → (50000, 9)
Applications: (100150, 8) → (100000, 8)
Repayments: (465982, 12) → (465782, 12)


In [29]:
# Check missing values

print("CUSTOMERS")
print(customers_clean.isnull().sum())

print("\nAPPLICATIONS")
print(applications_clean.isnull().sum())

print("\nCREDIT ASSESSMENTS")
print(credit.isnull().sum())

print("\nLOANS")
print(loans.isnull().sum())

print("\nREPAYMENTS")
print(repayments_clean.isnull().sum())

print("\nCOLLECTIONS")
print(collections.isnull().sum())

print("\nLOAN STATUS HISTORY")
print(loan_status_history.isnull().sum())

CUSTOMERS
customer_id          0
age                  0
gender               0
employment_type      0
monthly_income     150
customer_type        0
city                 0
region               0
customer_since       0
dtype: int64

APPLICATIONS
application_id             0
customer_id                0
application_date           0
requested_amount           0
approved_amount        45887
loan_type                  0
acquisition_channel      200
application_status         0
dtype: int64

CREDIT ASSESSMENTS
assessment_id         0
application_id        0
credit_score          0
debt_to_income      300
employment_years      0
risk_grade            0
assessment_date       0
dtype: int64

LOANS
loan_id                 0
application_id          0
disbursement_date       0
disbursement_amount     0
interest_rate          97
tenure_months           0
loan_status             0
dtype: int64

REPAYMENTS
repayment_id                0
loan_id                     0
installment_number          0
due_da

In [30]:
# Step 2A: Handle missing values


# 1. Customer monthly income
# Missing income will be replaced with the median income

income_median = customers_clean["monthly_income"].median()

customers_clean["monthly_income"] = (
    customers_clean["monthly_income"]
    .fillna(income_median)
)


# 2. Application approved amount
# Missing approved amount means the application was not approved

applications_clean["approved_amount"] = (
    applications_clean["approved_amount"]
    .fillna(0)
)


# 3. Acquisition channel
# Missing channel is kept as Unknown rather than guessing

applications_clean["acquisition_channel"] = (
    applications_clean["acquisition_channel"]
    .fillna("Unknown")
)


# 4. Debt-to-income ratio
# Replace missing values with median

dti_median = credit["debt_to_income"].median()

credit["debt_to_income"] = (
    credit["debt_to_income"]
    .fillna(dti_median)
)


# 5. Interest rate
# Replace missing values with median

interest_median = loans["interest_rate"].median()

loans["interest_rate"] = (
    loans["interest_rate"]
    .fillna(interest_median)
)


# 6. Payment date
# Keep missing values because missing payment date is meaningful

# 7. Promise-to-pay date
# Keep missing values because not every collection action has a PTP


print("Missing value treatment completed.")

Missing value treatment completed.


In [31]:
print("Customers missing income:",
      customers_clean["monthly_income"].isna().sum())

print("Applications missing approved amount:",
      applications_clean["approved_amount"].isna().sum())

print("Applications missing acquisition channel:",
      applications_clean["acquisition_channel"].isna().sum())

print("Credit missing DTI:",
      credit["debt_to_income"].isna().sum())

print("Loans missing interest rate:",
      loans["interest_rate"].isna().sum())

Customers missing income: 0
Applications missing approved amount: 0
Applications missing acquisition channel: 0
Credit missing DTI: 0
Loans missing interest rate: 0


In [32]:
# Step 3: Check invalid values

print("Negative monthly income:",
      (customers_clean["monthly_income"] < 0).sum())

print("Invalid credit scores:",
      (~credit["credit_score"].between(300, 850)).sum())

print("Negative amount paid:",
      (repayments_clean["amount_paid"] < 0).sum())

print("Invalid disbursement dates:",
      ((loans["disbursement_date"] < "2024-01-01") |
       (loans["disbursement_date"] > "2025-12-31")).sum())

Negative monthly income: 50
Invalid credit scores: 25
Negative amount paid: 60
Invalid disbursement dates: 30


In [33]:
# Fix negative monthly income

customers_clean.loc[
    customers_clean["monthly_income"] < 0,
    "monthly_income"
] = pd.NA

# Fill invalid/missing income with median of valid values
income_median = customers_clean["monthly_income"].median()

customers_clean["monthly_income"] = (
    customers_clean["monthly_income"].fillna(income_median)
)

print("Negative income remaining:",
      (customers_clean["monthly_income"] < 0).sum())

print("Missing income remaining:",
      customers_clean["monthly_income"].isna().sum())

Negative income remaining: 0
Missing income remaining: 0


In [34]:
# Fix invalid credit scores

credit.loc[
    ~credit["credit_score"].between(300, 850),
    "credit_score"
] = pd.NA

print("Invalid credit scores remaining:",
      (~credit["credit_score"].between(300, 850)).sum())

print("Missing credit scores:",
      credit["credit_score"].isna().sum())

Invalid credit scores remaining: 25
Missing credit scores: 25


In [35]:
print("Invalid non-missing credit scores:",
      ((credit["credit_score"].notna()) &
       (~credit["credit_score"].between(300, 850))).sum())

print("Missing credit scores:",
      credit["credit_score"].isna().sum())

Invalid non-missing credit scores: 0
Missing credit scores: 25


In [36]:
# Fix negative payment amounts

repayments_clean.loc[
    repayments_clean["amount_paid"] < 0,
    "amount_paid"
] = pd.NA

print("Negative amount_paid remaining:",
      (repayments_clean["amount_paid"] < 0).sum())

print("Missing amount_paid:",
      repayments_clean["amount_paid"].isna().sum())

Negative amount_paid remaining: 0
Missing amount_paid: 60


In [37]:
# Convert disbursement_date to datetime first

loans["disbursement_date"] = pd.to_datetime(
    loans["disbursement_date"],
    errors="coerce"
)

# Identify dates outside project period

invalid_date_mask = (
    (loans["disbursement_date"] < "2024-01-01") |
    (loans["disbursement_date"] > "2025-12-31")
)

print("Invalid dates found:", invalid_date_mask.sum())

# Set invalid dates to missing

loans.loc[invalid_date_mask, "disbursement_date"] = pd.NaT

print("Invalid dates remaining:",
      (
          (loans["disbursement_date"] < "2024-01-01") |
          (loans["disbursement_date"] > "2025-12-31")
      ).sum())

print("Missing disbursement dates:",
      loans["disbursement_date"].isna().sum())

Invalid dates found: 30
Invalid dates remaining: 0
Missing disbursement dates: 30


In [38]:
print(
    applications_clean["acquisition_channel"]
    .value_counts(dropna=False)
)

acquisition_channel
Mobile App           29948
Digital Marketing    19978
Direct               19781
Partner              17930
Referral             12043
Unknown                200
PARTNER                 40
mobile app              40
Digital marketing       40
Name: count, dtype: int64


In [39]:
# Standardize acquisition channel names

channel_mapping = {
    "PARTNER": "Partner",
    "mobile app": "Mobile App",
    "Digital marketing": "Digital Marketing"
}

applications_clean["acquisition_channel"] = (
    applications_clean["acquisition_channel"]
    .replace(channel_mapping)
)

# Verify

print(
    applications_clean["acquisition_channel"]
    .value_counts(dropna=False)
)

acquisition_channel
Mobile App           29988
Digital Marketing    20018
Direct               19781
Partner              17970
Referral             12043
Unknown                200
Name: count, dtype: int64


In [40]:
# Convert relevant dates to datetime

applications_clean["application_date"] = pd.to_datetime(
    applications_clean["application_date"],
    errors="coerce"
)

repayments_clean["due_date"] = pd.to_datetime(
    repayments_clean["due_date"],
    errors="coerce"
)

repayments_clean["payment_date"] = pd.to_datetime(
    repayments_clean["payment_date"],
    errors="coerce"
)


# Check 1: Payment before due date

payment_before_due = (
    repayments_clean["payment_date"].notna() &
    (repayments_clean["payment_date"] < repayments_clean["due_date"])
).sum()


# Check 2: Disbursement before application

loan_application_dates = loans.merge(
    applications_clean[["application_id", "application_date"]],
    on="application_id",
    how="left"
)

disbursement_before_application = (
    loan_application_dates["disbursement_date"].notna() &
    (loan_application_dates["disbursement_date"] <
     loan_application_dates["application_date"])
).sum()


# Check 3: Payment greater than amount due

payment_greater_than_due = (
    repayments_clean["amount_paid"].notna() &
    (repayments_clean["amount_paid"] >
     repayments_clean["amount_due"])
).sum()


print("Payment before due date:", payment_before_due)
print("Disbursement before application:", disbursement_before_application)
print("Payment greater than amount due:", payment_greater_than_due)

Payment before due date: 0
Disbursement before application: 0
Payment greater than amount due: 216062


In [41]:
# Investigate payments greater than amount due

overpayment_rows = repayments_clean[
    repayments_clean["amount_paid"].notna() &
    (repayments_clean["amount_paid"] > repayments_clean["amount_due"])
].copy()

print("Rows with payment > amount due:", len(overpayment_rows))

print("\nSample:")
print(
    overpayment_rows[
        ["loan_id", "installment_number",
         "amount_due", "amount_paid",
         "dpd", "overdue_amount"]
    ].head(10)
)

Rows with payment > amount due: 216062

Sample:
        loan_id  installment_number  amount_due  amount_paid  dpd  \
0   LOAN0000001                   1     5143.18      5236.93    0   
3   LOAN0000002                   2    22552.52     22553.93    0   
4   LOAN0000002                   3    22552.52     22874.51    0   
7   LOAN0000003                   3     8331.94      8378.20    0   
9   LOAN0000004                   2    65346.25     65688.80    0   
10  LOAN0000005                   1     5517.02      5595.05    0   
11  LOAN0000005                   2     5517.02      5571.04    0   
14  LOAN0000005                   5     5517.02      5618.41    0   
17  LOAN0000005                   8     5517.02      5588.11    0   
19  LOAN0000006                   1     3236.53      3246.78    0   

    overdue_amount  
0              0.0  
3              0.0  
4              0.0  
7              0.0  
9              0.0  
10             0.0  
11             0.0  
14             0.0  
17 

In [42]:
# Check DPD vs default flag

default_mismatch = (
    ((repayments_clean["dpd"] >= 90) &
     (repayments_clean["default_flag"] == 0))
    |
    ((repayments_clean["dpd"] < 90) &
     (repayments_clean["default_flag"] == 1))
).sum()

print("Default flag mismatches:", default_mismatch)

Default flag mismatches: 0


In [43]:
# Step 6: Validate relationships between tables

print(
    "Invalid customer references:",
    (~applications_clean["customer_id"].isin(
        customers_clean["customer_id"]
    )).sum()
)

print(
    "Invalid credit assessment references:",
    (~credit["application_id"].isin(
        applications_clean["application_id"]
    )).sum()
)

print(
    "Invalid loan references:",
    (~loans["application_id"].isin(
        applications_clean["application_id"]
    )).sum()
)

print(
    "Invalid repayment references:",
    (~repayments_clean["loan_id"].isin(
        loans["loan_id"]
    )).sum()
)

print(
    "Invalid collection references:",
    (~collections["loan_id"].isin(
        loans["loan_id"]
    )).sum()
)

print(
    "Invalid status history references:",
    (~loan_status_history["loan_id"].isin(
        loans["loan_id"]
    )).sum()
)

Invalid customer references: 0
Invalid credit assessment references: 0
Invalid loan references: 0
Invalid repayment references: 0
Invalid collection references: 0
Invalid status history references: 0


In [44]:
import os

# Create cleaned data folder
clean_folder = "FinNova_Digital_Lending_Clean"

os.makedirs(clean_folder, exist_ok=True)


# Export cleaned datasets

customers_clean.to_csv(
    f"{clean_folder}/customers_clean.csv",
    index=False
)

applications_clean.to_csv(
    f"{clean_folder}/applications_clean.csv",
    index=False
)

credit.to_csv(
    f"{clean_folder}/credit_assessments_clean.csv",
    index=False
)

loans.to_csv(
    f"{clean_folder}/loans_clean.csv",
    index=False
)

repayments_clean.to_csv(
    f"{clean_folder}/repayments_clean.csv",
    index=False
)

collections.to_csv(
    f"{clean_folder}/collections_clean.csv",
    index=False
)

loan_status_history.to_csv(
    f"{clean_folder}/loan_status_history_clean.csv",
    index=False
)

print("Clean datasets exported successfully.")
print(os.listdir(clean_folder))

Clean datasets exported successfully.
['applications_clean.csv', 'collections_clean.csv', 'credit_assessments_clean.csv', 'customers_clean.csv', 'loans_clean.csv', 'loan_status_history_clean.csv', 'repayments_clean.csv']
